In [ ]:
import logic.pulsed.pulse_objects as po
from logic.pulsed.sampling_functions import SamplingFunctions as SF
import time
import matplotlib.pyplot as plt
from tqdm import tqdm
pjl = pulsedjupyterlogic_AWG #Make sure the Logic is started in the Qudi manager!!!!
import pickle
import datetime
from logic import gwyfile as gwy
import numpy as np
from scipy.signal import find_peaks
from hardware.timetagger_counter import HWRecorderMode

### Measurement parameters for PODMR

In [ ]:
#measurement parameters
pi_pulse = 72e-9
power_0 = -20

#Information for running full PODMR
target_freq_0 = 2.80e9
mw_start = 2.78e9
mw_stop = 2.84e9
mw_step = 1e6

max_sweeps_podmr =150e3 #Total number of measured PODMR spectra
sweeps_per_run = 100 #Number of PODMR measuremnts for one run, e.g. measure 100 PODMR spectra
                    #and transfer data to the PC, than measure again (number of available bins on timetagger is limiting)

bin_width_s = pulsedsettingslogic.bin_width
record_length_s = pulsedsettingslogic.read_out_time
add_tt_read_out = pulsedsettingslogic.add_tt_read_out

retract = False

#additional information for save tag
tip_name = 'A-S07-29'
sample = 'YBCO_S1'
temperature = '82K'
b_field = '5mT_OOP'
contact = 'Z_lift_off'
extra = 'Vortex'

afm_scanner_logic.jupyter_meas_stop = False
afm_scanner_logic.jupyter_meas_pause = False

In [ ]:
force_ready = False
analysis_settings = podmrlogic.pulsed_analysis_settings
pjl.initialize_ensemble(laser_power_voltage = podmrlogic.laser_power_voltage, pi_pulse=pi_pulse, LO_freq_0=target_freq_0+100e6, target_freq_0=target_freq_0, power_0=power_0, set_up_measurement = False)
AWG_ensemble_list, AWG_sequence_step_list, PS_seq_name, tau_arr, alternating, freq_sweep = pjl.PODMR(mw_start, mw_stop, mw_step)
freq_points = len(tau_arr)
n_bins = 1+int((record_length_s + add_tt_read_out)/bin_width_s)
num_runs = int(max_sweeps_podmr/sweeps_per_run)
PODMR_raw_data = np.zeros([num_runs, freq_points*sweeps_per_run, n_bins])

# ret_val = time_tagger.configure_recorder(
#         mode=HWRecorderMode.GENERAL_PULSED,
#         params={'laser_pulses': freq_points*sweeps_per_run,
#                 'bin_width_s': bin_width_s,
#                 'record_length_s': record_length_s+add_tt_read_out,
#                 'max_counts': 1} )
pjl.fastcounter.configure(bin_width_s, record_length_s+add_tt_read_out, freq_points*sweeps_per_run)
pjl.fastcounter.pulsed.setMaxCounts(1)

pjl.mw.set_cw(frequency = pjl.LO_freq_0, power = pjl.power_0)
pjl.mw.cw_on()
pjl.AWG.pulser_on()

for i in range(num_runs):
    pjl.fastcounter.start_measure() #not sure if working without pjl.fastcounter.stop_measure()
    while True:
        if pjl.fastcounter.pulsed.ready() or force_ready:
            force_ready = False
            break
            
    PODMR_raw_data[i] = np.array(pjl.fastcounter.pulsed.getData(),dtype = 'int64')
    
pjl.AWG.pulser_off()
pjl.mw.off()
pjl.fastcounter.stop_measure()  

In [ ]:
#Analysis with fast scan mode/FFT
PODMR_raw_data_concatenate = PODMR_raw_data.reshape(num_runs*sweeps_per_run*freq_points, n_bins)
PODMR_data_concatenate, PODMR_err_concatenate, ref_data_concatenate, ref_time_concatenate = afm_scanner_logic.analyse_pulsed_meas(analysis_settings, PODMR_raw_data_concatenate, False)



#Analysis with sum of all PODMR spectra (normal way)
PODMR_raw_data_reshape = PODMR_raw_data.reshape(num_runs*sweeps_per_run, freq_points, n_bins)
PODMR_raw_data_sum = PODMR_raw_data_reshape.sum(axis=(0))
PODMR_data_sum, PODMR_err_sum, ref_data_sum, ref_time_sum = afm_scanner_logic.analyse_pulsed_meas(analysis_settings, PODMR_raw_data_sum, False)

fit = afm_scanner_logic._fitlogic.make_gaussian_fit(tau_arr,PODMR_data_sum,estimator=afm_scanner_logic._fitlogic.estimate_gaussian_dip)
lm,_ = fitlogic.make_gaussian_model()

#Plot Data
plt.rcParams.update({'font.size': 10})
fig = plt.figure(figsize = (5, 4))

ax1 = fig.add_subplot()

ax1.set_xlabel('Frequency (GHz)')
ax1.set_ylabel('Signal (a.u.)')

leg = f'Data'

ax1.errorbar(x=tau_arr/1e9, y=PODMR_data_sum,
                         yerr=PODMR_err_sum, fmt='-o',
                         capsize=3, capthick=0.9, color = 'blue',
                        elinewidth=1.2, markersize=3, linewidth=1.0, label = leg)
leg = f"Resonance frequency: {round(fit.params['center']/1e9,4)} GHz $\pm$ {round(fit.params['center'].stderr/1e9,4)} GHz"
ax1.plot(var_list_more*1e3, temp/data[0],'-', linewidth=1.5, label=leg, color = 'red')

plt.show()